# Predicción del test: binario (0 = nevus, 1 = lesión)

Carga un modelo ya entrenado (de la carpeta de resultados de un notebook de entrenamiento), predice el test y guarda el
CSV de la entrega: **una sola columna, 0 = nev y 1 = les, en el orden de la numeración de las imágenes**.

**Lo único que hay que tocar es la primera celda:**
- `RESULTS_DIR`: la carpeta de resultados del notebook de entrenamiento (p. ej. `ml/ml_results_binary_gridsearch`).
- `MODEL_FILE`: qué modelo de `RESULTS_DIR/models/` usar. La segunda celda los lista todos con su nombre:
  - `final_model_trainval.joblib`: el elegido, reentrenado con train + val (sección 9 del notebook de entrenamiento).
  - `best_model.joblib`: el mejor de la CV, entrenado solo con train.
  - `<modelo>.joblib`: cualquier otro (`xgboost.joblib`, `ens_xgb_svm.joblib`...), entrenado solo con train.
- `THRESHOLD`: `"auto"` usa el umbral guardado para ese modelo (si no hay, el de por defecto: 0.5 sobre la
  probabilidad, o 0 sobre `decision_function` en el SVM). También se puede poner un número.

Antes de predecir el test, comprueba el mismo modelo con el mismo umbral **en val**: si la accuracy cuadra con la del
notebook de entrenamiento, se ha cargado lo que toca.

Necesita `features_test_binary_final/features_test.csv` (notebooks 05 y 06) y las imágenes de `test_binary/`
(para comprobar que hay una predicción por imagen y ordenarlas).

In [7]:
import json
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, recall_score

warnings.filterwarnings("ignore")

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Binary").is_dir()), Path.cwd())
# ROOT = Path(r"C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG")    # o fíjala a mano

# ======================= LO QUE SE CAMBIA =======================
RESULTS_DIR = ROOT / "ml" / "ml_results_binary_gridsearch"   # carpeta de resultados del notebook de entrenamiento
MODEL_FILE = "ens_xgb_svm.joblib"                    # archivo de RESULTS_DIR/models/ (la celda siguiente los lista)
THRESHOLD = "auto"                                            # "auto" | un número (p. ej. 0.5)
# ================================================================

TEST_CSV = ROOT / "features_test_binary_final" / "features_test.csv"
TEST_IMG_DIR = ROOT / "test_binary"                 # imágenes originales del test (para no perder ninguna y ordenarlas)
OUT_CSV = ROOT / "submission" / "test_binary_predictions.csv"
CHECK_ON_VAL = True                                  # comprueba el modelo + umbral en val antes de predecir el test
SUBMIT_NAMES = {0: "nevus", 1: "lesión"}            # códigos de la entrega

MODEL_DIR = RESULTS_DIR / "models"
assert MODEL_DIR.is_dir(), f"No existe {MODEL_DIR}: ¿has ejecutado el notebook de entrenamiento con esa carpeta?"
print("ROOT:", ROOT)

ROOT: C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG


## 1. Modelos disponibles

In [8]:
def read_json(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


info_all = read_json(MODEL_DIR / "models_info.json")              # modelos entrenados con train ("Guardar los modelos")
info_final = read_json(MODEL_DIR / "final_model_trainval.json")   # modelo reentrenado con train + val (última sección)

file2name = {f: n for n, f in info_all.get("files", {}).items() if n != "best"}
if info_all.get("best_model"):
    file2name["best_model.joblib"] = f"{info_all['best_model']}  (best_model: el mejor de la CV)"
if info_final:
    file2name["final_model_trainval.joblib"] = f"{info_final['model']}  (reentrenado con train + val)"

print(f"Modelos disponibles en {MODEL_DIR} (pon el archivo en MODEL_FILE):\n")
for f in sorted(MODEL_DIR.glob("*.joblib")):
    mark = "  <-- elegido" if f.name == MODEL_FILE else ""
    print(f"  {f.name:<38s} {file2name.get(f.name, ''):<55s} {f.stat().st_size / 1e6:6.1f} MB{mark}")
assert (MODEL_DIR / MODEL_FILE).exists(), f"No existe {MODEL_DIR / MODEL_FILE}: elige uno de la lista"

Modelos disponibles en C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\ml\ml_results_binary_gridsearch\models (pon el archivo en MODEL_FILE):

  best_model.joblib                      Ens XGB+SVM  (best_model: el mejor de la CV)              21.9 MB
  decision_tree.joblib                   Decision Tree                                              0.0 MB
  ens_xgb_hgb_rf.joblib                  Ens XGB+HGB+RF                                            15.8 MB
  ens_xgb_hgb_svm.joblib                 Ens XGB+HGB+SVM                                           23.5 MB
  ens_xgb_svm.joblib                     Ens XGB+SVM                                               21.9 MB  <-- elegido
  ens_xgb_svm_logreg.joblib              Ens XGB+SVM+LogReg                                        21.9 MB
  final_model_trainval.joblib            Ens XGB+SVM  (reentrenado con train + val)                26.1 MB
  histgradientboosting.joblib            HistGradientBoosting                         

## 2. Cargar el modelo y el umbral

In [9]:
model = joblib.load(MODEL_DIR / MODEL_FILE)
is_final = MODEL_FILE == "final_model_trainval.joblib"
if is_final:
    MODEL_NAME = info_final["model"]
elif MODEL_FILE == "best_model.joblib":
    MODEL_NAME = info_all.get("best_model", MODEL_FILE)
else:
    MODEL_NAME = file2name.get(MODEL_FILE, MODEL_FILE)
TRAINED_ON = "train + val" if is_final else "train"
FEATS = (info_final if is_final else info_all).get("features") or info_all["features"]

has_proba = hasattr(model, "predict_proba")
THR_DEFAULT = 0.5 if has_proba else 0.0       # el que usa model.predict()


def score(X):
    """Puntuación de 'lesión' (clase 1): probabilidad, o decision_function si el modelo no da probabilidades (SVM)."""
    return model.predict_proba(X)[:, 1] if has_proba else model.decision_function(X)


thr_saved = read_json(RESULTS_DIR / "decision_threshold.json")
if THRESHOLD == "auto":
    if is_final and "threshold" in info_final:
        thr, thr_src = float(info_final["threshold"]), "final_model_trainval.json"
    elif thr_saved.get("model") == MODEL_NAME:
        thr, thr_src = float(thr_saved["threshold"]), "decision_threshold.json"
    else:
        thr, thr_src = THR_DEFAULT, "por defecto (no hay umbral guardado para este modelo)"
else:
    thr, thr_src = float(THRESHOLD), "fijado a mano"
DECISION_INFO = {"threshold": thr, "source": thr_src, "default": THR_DEFAULT}

print(f"Modelo:     {MODEL_NAME}  [{MODEL_FILE}]")
print(f"Entrenado:  con {TRAINED_ON} | {len(FEATS)} features")
print(f"Umbral:     {thr:.4f}  ({thr_src}; el de por defecto es {THR_DEFAULT})")

Modelo:     Ens XGB+SVM  [ens_xgb_svm.joblib]
Entrenado:  con train | 412 features
Umbral:     0.5000  (decision_threshold.json; el de por defecto es 0.5)


## 3. Comprobación en val
Mismo modelo y mismo umbral sobre val. Si el modelo se entrenó solo con train, la accuracy debe coincidir con la del
notebook de entrenamiento. Si es el reentrenado con train + val, saldrá más alta de lo real (ya ha visto val):
solo sirve para comprobar que todo cuadra.

In [10]:
if CHECK_ON_VAL:
    val_csv = next((ROOT / d / "features_val.csv" for d in ("features_binary_final", "features_final")
                    if (ROOT / d / "features_val.csv").exists()), None)
    if val_csv is None:
        print("No encuentro features_val.csv: se omite la comprobación")
    else:
        va = pd.read_csv(val_csv)
        yv = va["label"].to_numpy()
        pv = (score(va[FEATS]) >= thr).astype(int)
        print(f"Val ({len(yv)} imágenes, {val_csv.parent.name}/): accuracy {accuracy_score(yv, pv):.4f} | "
              f"bal_acc {balanced_accuracy_score(yv, pv):.4f} | recall lesión {recall_score(yv, pv):.4f}\n")
        print(pd.DataFrame(confusion_matrix(yv, pv), index=["real nevus", "real lesión"],
                           columns=["pred nevus", "pred lesión"]).to_string())
        if is_final:
            print("\nOJO: este modelo se entrenó también con val -> la cifra sale optimista (solo comprueba que todo cuadra).")

Val (3796 imágenes, features_binary_final/): accuracy 0.8778 | bal_acc 0.8776 | recall lesión 0.8708

             pred nevus  pred lesión
real nevus         1708          223
real lesión         241         1624


## 4. Predicción del test (una fila por imagen, en orden)

In [11]:
def image_number(name):
    nums = re.findall(r"\d+", str(name))
    return int(nums[-1]) if nums else None


te = pd.read_csv(TEST_CSV)
missing_cols = [c for c in FEATS if c not in te.columns]
assert not missing_cols, f"Al CSV del test le faltan {len(missing_cols)} features (p. ej. {missing_cols[:5]})"
te["name"] = te["name"].astype(str)
assert te["name"].is_unique, "hay nombres repetidos en el CSV del test"
n_nan = int(te[FEATS].isna().any(axis=1).sum())
print(f"Test: {len(te)} imágenes con features | {n_nan} con algún NaN (se imputan con la mediana de train, como en val)")

# lista completa de imágenes del test, para que no falte ninguna fila
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
if TEST_IMG_DIR.is_dir():
    all_names = [f.stem for f in TEST_IMG_DIR.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT]
    print(f"Imágenes en {TEST_IMG_DIR.name}/: {len(all_names)}")
else:
    print(f"AVISO: no encuentro {TEST_IMG_DIR}: se usan solo las imágenes del CSV (no se puede comprobar si falta alguna)")
    all_names = list(te["name"])

# orden = numeración de los archivos de imagen
nums = [image_number(n) for n in all_names]
if any(n is None for n in nums) or len(set(nums)) != len(nums):
    print("AVISO: no todos los nombres tienen un número único -> se ordenan alfabéticamente")
    ORDER = sorted(all_names)
else:
    ORDER = [n for _, n in sorted(zip(nums, all_names))]

pred_by_name = pd.Series((score(te[FEATS]) >= thr).astype(int), index=te["name"])

missing = [n for n in ORDER if n not in pred_by_name.index]
extra = sorted(set(pred_by_name.index) - set(ORDER))
if missing:
    fallback = int(pred_by_name.mode()[0])
    print(f"AVISO: {len(missing)} imágenes del test NO tienen features (falló el preprocesado o la extracción): "
          f"{missing[:10]}\n       -> se les asigna la clase más frecuente ({fallback}). Mejor revisarlas y extraerlas.")
    pred_by_name = pd.concat([pred_by_name, pd.Series(fallback, index=missing)])
if extra:
    print(f"AVISO: {len(extra)} nombres del CSV no están entre las imágenes del test (no se entregan): {extra[:10]}")

pred = pred_by_name.reindex(ORDER).astype(int)
print(f"\n{len(pred)} predicciones, en orden | primeras: {list(pred.index[:3])} ... última: {pred.index[-1]}")
print("recuento:", {f"{k} ({SUBMIT_NAMES[k]})": int((pred == k).sum()) for k in sorted(SUBMIT_NAMES)})

Test: 6340 imágenes con features | 0 con algún NaN (se imputan con la mediana de train, como en val)
AVISO: no encuentro C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\test_binary: se usan solo las imágenes del CSV (no se puede comprobar si falta alguna)

6340 predicciones, en orden | primeras: ['xxx00001', 'xxx00002', 'xxx00003'] ... última: xxx06340
recuento: {'0 (nevus)': 3245, '1 (lesión)': 3095}


## 5. Guardar el CSV de la entrega
- `submission/test_binary_predictions.csv`: **lo que se entrega** (una columna, sin cabecera ni índice).
- `..._con_nombres.csv`: lo mismo con el nombre de cada imagen, para revisar el orden (no se entrega).
- `.json`: qué modelo, umbral y carpeta se usaron.

Si el profesor quiere cabecera, cambia `header=False` por `header=["label"]`.

In [12]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
pred.to_csv(OUT_CSV, index=False, header=False)                     # LO QUE SE ENTREGA: una sola columna, sin cabecera
check_csv = OUT_CSV.with_name(OUT_CSV.stem + "_con_nombres.csv")    # solo para revisar (no se entrega)
pred.rename("pred").rename_axis("name").to_frame().assign(clase=pred.map(SUBMIT_NAMES).values).to_csv(check_csv)
OUT_CSV.with_suffix(".json").write_text(json.dumps(
    {"model": MODEL_NAME, "model_file": MODEL_FILE, "results_dir": str(RESULTS_DIR), "trained_on": TRAINED_ON,
     "decision": DECISION_INFO, "codes": SUBMIT_NAMES, "n_images": len(pred)},
    indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("PARA ENTREGAR:", OUT_CSV)
print("para revisar: ", check_csv, "\n")
print(pd.read_csv(OUT_CSV, header=None).head(5).to_string(header=False))

PARA ENTREGAR: C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\submission\test_binary_predictions.csv
para revisar:  C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\submission\test_binary_predictions_con_nombres.csv 

0  0
1  0
2  1
3  1
4  1
